# ConSeMa + AG-MPS on Google Colab

Reproduction of *Mossina & Friedrich, "Conformal Prediction for Image Segmentation Using Morphological
Prediction Sets" (MICCAI 2025)* plus the **AG-MPS extension** (geodesic image-guided margins E1,
size-adaptive scores E2, exact conformal quantile E3). See `EXTENSION.md`.

**Before you start:** `Runtime → Change runtime type → T4 GPU`.

Two ways to use this notebook:
* **A. Only view the results** (≈5 min). Upload `consema_project.zip` built with
  `scripts/make_colab_bundle.sh --demo`, run sections 1 and 7.
* **B. Re-run everything** (data download → UniverSeg predictions → 36-seed experiments → viewer).
  Run sections 1–7. Colab has only 2 CPU cores, so set `SEEDS` in section 5 to fewer seeds for a quick run.

## 1. Upload and unpack the project

In [ ]:
import os, zipfile
from google.colab import files
if not os.path.exists('/content/consema'):
    up = files.upload()                       # choose consema_project.zip
    zipfile.ZipFile(next(iter(up))).extractall('/content/consema')
%cd /content/consema
!ls

In [ ]:
# viewer + extension dependencies (torch, numpy, scikit-image, pandas are preinstalled on Colab)
!pip install -q "gradio>=4.44,<6" einops nibabel tabulate gdown python-dotenv
!pip install -q -e . --no-deps

## 2. Datasets and models (only for option B)

In [ ]:
%%bash
set -e
cd /content/consema
mkdir -p benchmarks/data benchmarks/weights benchmarks/polyps_precomputed
# UniverSeg (pinned commit used by the paper)
[ -d benchmarks/universeg ] || git clone -q https://github.com/JJGO/UniverSeg benchmarks/universeg
cd benchmarks/universeg && git checkout -q 833a0c34c65e38d675e21bd48ddec6797cc03259 && touch example_data/__init__.py && cd ../..
pip install -q -e benchmarks/universeg --no-deps
curl -sSL https://github.com/JJGO/UniverSeg/releases/download/weights/universeg_v1_nf64_ss64_STA.pt -o benchmarks/weights/universeg_v1_nf64_ss64_STA.pt
# WBC
[ -d benchmarks/data/universeg_wbc ] || git clone -q https://github.com/zxaoyou/segmentation_WBC.git benchmarks/data/universeg_wbc
# OASIS 2D (moved to HuggingFace; the old UniverSeg URL is dead)
mkdir -p benchmarks/data/universeg_oasis
curl -sSL "https://huggingface.co/datasets/DalcaLab/neurite-oasis/resolve/main/neurite-oasis.2d.v1.0.tar?download=true" -o benchmarks/data/universeg_oasis/oasis2d.tar
echo "c9ae5864f250c7e4b8d83a104e51ae8e  benchmarks/data/universeg_oasis/oasis2d.tar" | md5sum -c
tar xf benchmarks/data/universeg_oasis/oasis2d.tar -C benchmarks/data/universeg_oasis
# Polyps + precomputed PraNet predictions (1.3 GB; gzip despite the .xz name)
python -m gdown 1h7S6N_Rx7gdfO3ZunzErZy6H7620EbZK -O benchmarks/polyps_precomputed/data_polyps.tar.xz
tar -xzf benchmarks/polyps_precomputed/data_polyps.tar.xz -C benchmarks/polyps_precomputed data/polyps
rm benchmarks/polyps_precomputed/data_polyps.tar.xz
# paths used by the official code
P=/content/consema
printf "PROJECT_DIR=$P\nEXPERIMENTS_DIR=$P/experiments\nDATA_DIR=$P/benchmarks\nPOLYPS_NPZ=$P/benchmarks/polyps_precomputed/data/polyps\nCONFIGS_EXPERIMENTS=$P/experiments/configs\nRESULTS_DIR=$P/results\n" > .env
du -sh benchmarks/*

## 3. Tests: extension == paper when β = γ = 0, and coverage is valid

In [ ]:
!python -m pytest tests -q 2>&1 | tail -3

## 4. UniverSeg predictions on GPU (36 seeds, paper protocol)

In [ ]:
import torch; print('GPU:', torch.cuda.is_available())
!python experiments/extension/cache_predictions.py --dataset wbc
!python experiments/extension/cache_predictions.py --dataset oasis

## 5. Experiments: paper reproduction + AG-MPS
Full run = 36 seeds (same as the paper). On Colab's 2 CPUs this takes a while; use e.g. 10 seeds for a quick run.

In [ ]:
SEEDS = ' '.join(str(s) for s in range(1989, 2025))      # all 36 seeds
# SEEDS = ' '.join(str(s) for s in range(1989, 1999))    # quick: 10 seeds
for ds in ['wbc', 'oasis', 'polyps']:
    !python experiments/extension/run_extension.py --dataset {ds} --seeds {SEEDS}

In [ ]:
!python experiments/extension/analyze_results.py | head -40
!python experiments/extension/qualitative.py --dataset wbc --kind region --beta 64 --gamma 1.0 --alpha 0.05 --tau 0.99 --tag _blackbox
!python experiments/extension/qualitative.py --dataset oasis --kind edge --beta 32 --gamma 0 --alpha 0.05 --tau 0.99 --n 3 --tag _blackbox
!python experiments/extension/qualitative.py --dataset polyps --kind sigmoid --beta 64 --gamma 1.0 --alpha 0.05 --tau 0.99 --tag _graybox

## 6. (optional) Official paper script, unchanged method

In [ ]:
# writes results/results_polyps_fixed_disk_<timestamp>.csv (same format as results/results/)
!python experiments/experiments.py --dataset polyps

## 7. Results viewer (tables, figures, live explorer)

In [ ]:
import os
if not os.path.exists('demo/polyps.npz'):            # build demo data if it was not in the zip
    !python experiments/extension/make_demo_bundle.py
!python app/app.py --share       # open the *.gradio.live link printed below; stop the cell to quit

## 8. Download results

In [ ]:
!zip -qr results_extension.zip results/extension figures/extension
from google.colab import files; files.download('results_extension.zip')